# 02 — Image-based Molecular Embeddings
**Section A** generates 2D molecular structure images (PNG) from an SDF file.
**Section B** extracts rotation-equivariant image embeddings using a P4-equivariant
lifting convolution built on top of a ResNet50 backbone pre-trained on ImageNet.
The model is used as a fixed feature extractor.

**Output:** CSV with shape `(N_molecules, 12544)`.

## Configuration

In [ ]:
INPUT_SDF  = "data/molecules.sdf"              # path to input SDF file
IMAGES_DIR = "data/images/"                    # directory where PNGs will be saved
OUTPUT_CSV = "data/image_embeddings.csv"       # path to output CSV
REFERENCE_SDF     = "data/adagrasib.sdf"       # path to reference molecule SDF file
REFERENCE_CSV_OUT = "data/adagrasib_image.csv" # path to reference molecule output CSV file
IMAGE_SIZE = (400, 400)  # PNG size in pixels
BATCH_SIZE = 16          # inference batch size

## Dependencies

In [ ]:
# pip install rdkit torch torchvision pillow pandas numpy

---
## Section A — 2D Image Generation
Generates one PNG per molecule from the SDF file using RDKit.

### A.1 Load molecules from SDF

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
import os

os.makedirs(IMAGES_DIR, exist_ok=True)

supplier = Chem.SDMolSupplier(INPUT_SDF)
mols     = list(supplier)
valid    = [(i, mol) for i, mol in enumerate(mols) if mol is not None]
invalid  = [i for i, mol in enumerate(mols) if mol is None]

print(f"Total entries   : {len(mols)}")
print(f"Valid molecules : {len(valid)}")
if invalid:
    print(f"Invalid (skipped): {invalid}")

### A.2 Generate and save 2D images

In [ ]:
image_index = []  # keeps track of (dataset_index, image_path) for alignment

for i, mol in valid:
    AllChem.Compute2DCoords(mol)

    # Use MERGED_SDF_INDEX as filename if available, otherwise use row index
    name = mol.GetProp('MERGED_SDF_INDEX') if mol.HasProp('MERGED_SDF_INDEX') else str(i)
    name = "".join(c if c.isalnum() or c in ('_', '-') else '_' for c in name)

    out_path = os.path.join(IMAGES_DIR, f"{name}.png")
    Draw.MolToFile(mol, out_path, size=IMAGE_SIZE)
    image_index.append((i, out_path))

print(f"Generated {len(image_index)} images in: {IMAGES_DIR}")

---
## Section B — Image Embedding Extraction
Extracts 12544-dimensional embeddings using a P4-equivariant ResNet50 feature extractor.

### B.1 P4-equivariant architecture
The standard ResNet50 first convolutional layer is replaced by a P4 lifting convolution,
which applies the same kernel at 4 discrete rotations (0°, 90°, 180°, 270°), yielding
rotation-equivariant feature maps. Weights are initialized from pre-trained ResNet50 (ImageNet).
Group pooling at the end collapses the orientation dimension, producing rotation-invariant embeddings.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision


def rotate_kernels_90(w: torch.Tensor, k: int) -> torch.Tensor:
    """Rotate convolutional kernels by 90 degrees k times."""
    return torch.rot90(w, k=k, dims=(-2, -1))


class P4LiftingConv2d(nn.Module):
    """
    Lifting convolution R^2 -> p4.
    Applies the same kernel at 4 discrete rotations (0, 90, 180, 270 degrees)
    and stacks outputs along a group dimension: output shape (B, 4, Cout, H', W').
    """
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=0, bias=False):
        super().__init__()
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        self.stride  = stride
        self.padding = padding
        self.weight  = nn.Parameter(torch.empty(out_channels, in_channels, *self.kernel_size))
        nn.init.kaiming_normal_(self.weight, mode='fan_out', nonlinearity='relu')
        if bias:
            self.bias_param = nn.Parameter(torch.zeros(out_channels))
        else:
            self.register_parameter('bias_param', None)

    def forward(self, x):
        outs = []
        for k in range(4):
            wk = rotate_kernels_90(self.weight, k=k)
            outs.append(F.conv2d(x, wk, bias=self.bias_param,
                                 stride=self.stride, padding=self.padding))
        return torch.stack(outs, dim=1)  # (B, 4, Cout, H', W')


class P4ResNet50FeatureExtractor(nn.Module):
    """
    Fixed feature extractor based on P4-equivariant ResNet50.
    Produces embeddings of shape (B, 12544) = (B, 64 * 14 * 14).
    No training is performed: weights are loaded from pre-trained ResNet50.
    """
    def __init__(self, gn_groups=32):
        super().__init__()
        resnet = torchvision.models.resnet50(
            weights=torchvision.models.ResNet50_Weights.DEFAULT
        )
        conv1 = resnet.conv1
        self.gconv1 = P4LiftingConv2d(
            in_channels=conv1.in_channels,
            out_channels=conv1.out_channels,
            kernel_size=conv1.kernel_size,
            stride=conv1.stride,
            padding=conv1.padding,
            bias=(conv1.bias is not None),
        )
        with torch.no_grad():
            self.gconv1.weight.copy_(conv1.weight.data)
        self.gn      = nn.GroupNorm(num_groups=gn_groups, num_channels=conv1.out_channels)
        self.relu    = nn.ReLU(inplace=True)
        self.maxpool = resnet.maxpool

    def forward(self, x):
        y = self.gconv1(x)                         # (B, 4, 64, 112, 112)
        B, G, C, H, W = y.shape
        y = self.gn(self.relu(y.view(B*G, C, H, W)))  # GN + ReLU per orientation
        y = self.maxpool(y)                        # (B*4, 64, 56, 56)
        y = y.view(B, G, C, 56, 56).max(dim=1).values  # group pooling: (B, 64, 56, 56)
        y = F.avg_pool2d(y, kernel_size=4, stride=4)   # (B, 64, 14, 14)
        return y.flatten(1)                        # (B, 12544)


print('Architecture defined.')

### B.2 Load images

In [ ]:
import glob
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

# Load images in sorted order (matches the order of image_index from Section A)
files = sorted([
    p for p in glob.glob(os.path.join(IMAGES_DIR, '**', '*'), recursive=True)
    if os.path.splitext(p.lower())[1] in EXTS
])
print(f'Images found: {len(files)}')


class ImageListDataset(Dataset):
    def __init__(self, files, transform=None):
        self.files     = files
        self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.files[idx]

### B.3 Extract embeddings

In [ ]:
device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

preprocess = torchvision.models.ResNet50_Weights.DEFAULT.transforms()
dataset    = ImageListDataset(files, transform=preprocess)
loader     = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = P4ResNet50FeatureExtractor().to(device)
model.eval()

X_chunks = []
with torch.no_grad():
    for imgs, _ in loader:
        X_chunks.append(model(imgs.to(device)).cpu().numpy())

X_all = np.concatenate(X_chunks, axis=0)
print(f'Embedding matrix shape: {X_all.shape}')  # (N, 12544)

### B.4 Save to CSV

In [ ]:
import pandas as pd

df = pd.DataFrame(X_all, columns=[f'f_{i}' for i in range(X_all.shape[1])])
df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}  |  shape: {df.shape}')

---
## Section C — Reference Molecule Image Embedding


### C.1 Generate reference image

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

supplier_ref = Chem.SDMolSupplier(REFERENCE_SDF)
mol_ref = [mol for mol in supplier_ref if mol is not None]
assert len(mol_ref) == 1

AllChem.Compute2DCoords(mol_ref[0])
ref_img_path = os.path.join(IMAGES_DIR, "reference.png")
Draw.MolToFile(mol_ref[0], ref_img_path, size=IMAGE_SIZE)
print(f"Reference image saved: {ref_img_path}")

### C.2 Extract reference embedding

In [ ]:
ref_dataset = ImageListDataset([ref_img_path], transform=preprocess)
ref_loader  = DataLoader(ref_dataset, batch_size=1, shuffle=False, num_workers=0)

with torch.no_grad():
    for imgs, _ in ref_loader:
        ref_emb = model(imgs.to(device)).cpu().numpy()

df_ref = pd.DataFrame(ref_emb, columns=[f'f_{i}' for i in range(ref_emb.shape[1])])
df_ref.to_csv(REFERENCE_CSV_OUT, index=False)
print(f"Saved: {REFERENCE_CSV_OUT}  |  shape: {df_ref.shape}")